# اليوم 4، المعمل 6: تقسيم العملاء دون ملصقات

K-Means يعيد مجموعات حتى للضوضاء. لذلك نحتاج قياسًا، ملفات تعريف قابلة للتسمية، واختبار فائدة خارجي.

In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").exists())
sys.path.insert(0, str(ROOT / "src"))
DATA = ROOT / "data" / "raw"
RANDOM_STATE = 42


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from manafeth.data import load_customers

df = load_customers(DATA)
features = ["orders_per_month", "avg_basket_sar", "days_since_last_order", "distinct_categories", "promo_usage_rate"]
X = df[features].fillna(df[features].median()).copy()
X[["orders_per_month", "avg_basket_sar", "days_since_last_order"]] = np.log1p(X[["orders_per_month", "avg_basket_sar", "days_since_last_order"]])
scaled = StandardScaler().fit_transform(X)


## اختيار k

جرّب k من 2 إلى 8 على عينة 10,000 صف لتسريع silhouette. لا تختَر أعلى رقم آليًا. افحص سهولة تسمية المجموعات.

In [ ]:
sample = scaled[:10000]
rows = []
for k in range(2, 9):
    labels = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit_predict(sample)
    rows.append({"k": k, "silhouette": silhouette_score(sample, labels, sample_size=3000, random_state=RANDOM_STATE)})
display(pd.DataFrame(rows))

k = int(pd.DataFrame(rows).sort_values("silhouette", ascending=False).iloc[0].k)
kmeans = KMeans(n_clusters=k, n_init=20, random_state=RANDOM_STATE)
labels = kmeans.fit_predict(scaled)
profile = df.assign(cluster=labels).groupby("cluster").agg(size=("customer_id", "size"), orders_pm=("orders_per_month", "mean"), basket=("avg_basket_sar", "mean"), recency=("days_since_last_order", "mean"), churn_rate=("churned_30d", "mean"))
display(profile.round(2))
points = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(scaled)
plt.scatter(points[::10, 0], points[::10, 1], c=labels[::10], s=6, alpha=.4, cmap="tab10")
plt.xlabel("PC1"); plt.ylabel("PC2"); plt.title("Customer segments in a 2D PCA view"); plt.show()


## الملف التعريفي والخريطة

درّب KMeans بالقيمة المختارة. اعرض متوسطات الخصائص والحجم ومعدل churn لكل مجموعة، ثم استخدم PCA للرسم فقط.

In [ ]:
# أُنجزت خطوات هذا القسم في خلية الحل السابقة.


**ناتج التسليم:** جدول مجموعات بأسماء عربية عملية. الاسم يصف السلوك ولا يحكم على الشخص.